# 02 · Evaluate the prior

Sample `z ~ N(0, I)`, decode, and sanity-check reconstructions of held-out
slices. A good prior produces plausible knee images and reconstructs real
slices closely.

In [ ]:
import jax, jax.numpy as jnp, matplotlib.pyplot as plt
import numpy as np
from mrigen.train_vae import load_model
from mrigen.data import FastMRISlices
from mrigen import viz

vae = load_model('../checkpoints/vae_128.eqx', latent_dim=128)

# Held-out volume: the prior is judged on slices it never saw. `ds_train` is
# only here for the comparison in the autoencode cell.
ds = FastMRISlices('../data/processed', split='test')
ds_train = FastMRISlices('../data/processed', split='train')
print(f"test: {len(ds)} slices from {ds.volumes}")
print(f"train: {len(ds_train)} slices from {ds_train.volumes}")

### Prior samples

In [ ]:
# Eight draws from the prior, on the same [0, 1] scale as the real images so
# the contrast is not flattered by per-panel autoscaling.
key = jax.random.PRNGKey(0)
zs = jax.random.normal(key, (8, vae.latent_dim))
samples = jax.vmap(vae.decoder)(zs)

fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for a, s in zip(ax.ravel(), samples):
    viz.show_image(s, a, vmin=0, vmax=1)
fig.suptitle('samples from the prior:  z ~ N(0, I),  x = decoder(z)', fontsize=13)
fig.tight_layout()
fig.savefig('prior_samples.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"sample intensity: mean {float(samples.mean()):.3f}  std {float(samples.std()):.3f}  "
      f"max {float(samples.max()):.3f}")
print(f"real slices:      mean {ds.slices.mean():.3f}  std {ds.slices.std():.3f}  "
      f"max {ds.slices.max():.3f}")
print("\nIf the sample statistics are far from the real ones, z ~ N(0, I) is not where\n"
      "the encoder actually put the data — and MAP/NUTS, which use exactly that prior,\n"
      "will be fighting the decoder.")

### Autoencode a real slice  *(needs reparameterise TODO)*

In [ ]:
from mrigen import metrics
from mrigen.models.vae import reparameterise

# Autoencoding is the *ceiling* on MAP: a reconstruction is some decoder output,
# so it can never beat the best the decoder can do on this slice. Decode the
# posterior mean (z = mu) for the deterministic version, and a sampled z to see
# how much the posterior spread costs.
def autoencode_report(dataset, label, n=8):
    idx = np.linspace(0.25, 0.75, n) * (len(dataset) - 1)
    det, sampled, kls = [], [], []
    for i in idx.round().astype(int):
        xi = jnp.asarray(dataset[i])
        mu, logvar = vae.encoder(xi)
        det.append(metrics.psnr(np.asarray(xi), np.asarray(vae.decoder(mu)), data_range=1.0))
        z = reparameterise(mu, logvar, jax.random.PRNGKey(int(i)))
        sampled.append(metrics.psnr(np.asarray(xi), np.asarray(vae.decoder(z)), data_range=1.0))
        kls.append(float(-0.5 * jnp.sum(1 + logvar - mu**2 - jnp.exp(logvar))))
    print(f"{label:<6} autoencode PSNR: {np.mean(det):5.2f} dB (z = mu)   "
          f"{np.mean(sampled):5.2f} dB (z sampled)   "
          f"KL = {np.mean(kls):7.1f} nats  ({np.mean(kls) / vae.latent_dim:.2f}/dim)")

autoencode_report(ds_train, 'train')
autoencode_report(ds, 'test')
print("\nKL is the 'rate' — how much information z carries about x.\n"
      "  KL ~ 0        => posterior collapse: z says nothing, every z decodes to the\n"
      "                   same mean image, and the prior is useless for reconstruction.\n"
      "  KL very large => the latent is far outside N(0, I), so prior samples look wrong\n"
      "                   even though autoencoding is sharp.\n"
      "A healthy 128-d latent here sits in the tens-to-hundreds of nats.")

# And the picture: a held-out slice, its reconstruction, and the error.
x = jnp.asarray(ds[len(ds) // 2])
mu, logvar = vae.encoder(x)
viz.panel(x, vae.decoder(mu))
plt.show()

### The prior's own scorecard: quality, diversity, speed

These three numbers are the "prior intrinsics" row of the Friday table, and the fair way to compare
*priors* (VAE vs diffusion vs power spectrum) separately from *reconstructions*.

- **Quality:** do the samples above look like knees? (Eyeball, then the checklist in notebook 06.)
- **Diversity:** mean pairwise (1 − SSIM) over samples — 0 means every sample is the same image.
- **Speed:** seconds per sample; it matters once you compare with a diffusion prior.

In [ ]:
import time
from mrigen import metrics

zs = jax.random.normal(jax.random.PRNGKey(1), (16, vae.latent_dim))
decode_batch = jax.jit(jax.vmap(vae.decoder))
decode_batch(zs).block_until_ready()                      # warm-up (JIT)
t0 = time.perf_counter(); samples = decode_batch(zs).block_until_ready(); dt = time.perf_counter() - t0
print(f"diversity (mean pairwise 1-SSIM over 16 samples): {metrics.diversity(np.asarray(samples)):.3f}")
print(f"speed: {1e3 * dt / len(zs):.2f} ms per sample")

### Latent interpolation

Encode two real slices, walk a straight line between their latent codes, decode. A good prior morphs
one knee into another through *plausible* knees; a bad one passes through mush.

In [ ]:
x0, x1 = jnp.asarray(ds[0]), jnp.asarray(ds[len(ds) // 2])
z0, z1 = vae.encoder(x0)[0], vae.encoder(x1)[0]            # the latent means
ts = jnp.linspace(0, 1, 6)
path = jax.vmap(lambda t: vae.decoder(z0 + t * (z1 - z0)))(ts)
fig, ax = plt.subplots(1, 6, figsize=(18, 3))
for a, img, t in zip(ax, path, ts):
    viz.show_image(img, a, title=f"t = {float(t):.1f}")
plt.show()